In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import pandas as pd
import seaborn as sns
import glob
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
covid_path=r"C:\Users\Nandy\Downloads\archive (2)\COVID-19_Radiography_Dataset\COVID\images"
normal_path=r"C:\Users\Nandy\Downloads\archive (2)\COVID-19_Radiography_Dataset\Normal\images"

In [ ]:
IMG_SIZE=(150,150)
BATCH_SIZE=32

In [ ]:
covid_images=glob.glob(covid_path+r"\*.png")
print("Number of covid images:",len(covid_images))
normal_images=glob.glob(normal_path+r"\*.png")
print("Number of normal images:",len(normal_images))

In [ ]:
covid_df=pd.DataFrame({"filename":covid_images,"label":1})

normal_df=pd.DataFrame({"filename":normal_images,"label":0})

df=pd.concat([covid_df,normal_df],ignore_index=True)

In [ ]:
print("\nDataset shape:")
print(df.shape)
print("\nDataset head:")
print(df.head())
print("\nDataset tail:")
print(df.tail())
print(df.isnull().sum())

In [ ]:
print("\n class Distribution:")
print(df["label"].value_counts())

print("\n class Names:")
print(
    df["label"].value_counts().rename(
        index={0:"NORMAL",1:"COVID"}))

In [ ]:

# SHUFFLE DATA
df = df.sample(frac=1,random_state=42
).reset_index(drop=True)

train_df, test_df = train_test_split(df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"])

print("\nTraining Images:",len(train_df))
print("Testing Images:",len(test_df))

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.20)

In [ ]:
# TEST DATA GENERATOR
test_datagen = ImageDataGenerator(
    rescale=1./255)


In [ ]:
#TRAINING DATA GENERATOR
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="training",
    shuffle=True)


In [ ]:

validation_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="validation",
    shuffle=False)

In [ ]:
test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False)

In [ ]:
# DISPLAY GENERATOR INFORMATION
print("\nTraining Samples:",train_generator.samples)
print("Validation Samples:",validation_generator.samples)
print("Testing Samples:",test_generator.samples)

In [ ]:
images,labels=next(train_generator)
plt.figure(figsize=(12,8))
for i in range(18):
    plt.subplot(9,9,i+1)
    plt.imshow(images[i])
    if labels[i]==1:
        plt.title("Covid")
    else:
        plt.title("Normal")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model=Sequential([
    
    Conv2D(32,(3,3),activation="relu",
       input_shape=(150,150,3)),
    MaxPooling2D(2,2),

     Conv2D(64,(3,3),activation="relu"),
     MaxPooling2D(2,2),

     Conv2D(128,(3,3),activation="relu"),
     MaxPooling2D(2,2),

    Flatten(),

    Dense(128,activation="relu"),

    Dropout(0.5),

    Dense(1,activation="sigmoid")])

In [ ]:
model.summary()

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=['accuracy']
)

In [ ]:
history=model.fit(train_generator,validation_data=validation_generator,epochs=3)
test_loss,test_accuracy=model.evaluate(test_generator)
print("TEST RESULTS")
print("TEST Loss:,test_loss")
print("Test Accuracy:",test_accuracy)

In [ ]:
test_generator.reset()
predictions = model.predict(test_generator)

predicted_classes = (predictions >= 0.5).astype(int).flatten()

image_path = covid_images[0]
print("\nImage Used For Prediction:")
print(image_path)

img = tf.keras.utils.load_img(image_path,
    target_size=IMG_SIZE)


img_array = tf.keras.utils.img_to_array(img)


img_array = np.expand_dims(img_array,axis=0)


img_array = img_array / 255.0


prediction = model.predict(img_array)
probability = prediction[0][0]
print("\nPrediction Probability:",probability)


plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
if probability >= 0.5:
    print("Prediction: COVID")
    plt.title("Prediction: COVID")
else:
    print("Prediction: NORMAL")
    plt.title("Prediction: NORMAL")
plt.show()